In [1]:
import numpy as np
import torch
import clip
from transformers import AutoImageProcessor, AutoModel
from qdrant_client import QdrantClient, models
from pymilvus import MilvusClient, DataType
import pandas as pd
from PIL import Image
import os
import time
import random

os.getpid()

/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/clip/clip.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging
/home/inna/.cache/pypoetry/virtualenvs/sneakersearch-z_mDpiHd-py3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


763905

In [2]:
#поднимаем ВБД с маппингом в корень прокта
# в корне проекта выполнить 

#docker run -p 6333:6333 -p 6334:6334 -v "$(pwd)/qdrant_storage:/qdrant/storage" qdrant/qdrant

# инициализация клиента ВБД
client = QdrantClient("http://localhost:6333")

device = "cuda" if torch.cuda.is_available() else "cpu"
device

'cuda'

In [ ]:
# client.recreate_collection(
#         collection_name="sneakers",
#         vectors_config=models.VectorParams(size=512, distance=models.Distance.COSINE),
#     )

/tmp/ipykernel_15972/1366355860.py:1: DeprecationWarning: `recreate_collection` method is deprecated and will be removed in the future. Use `collection_exists` to check collection existence and `create_collection` instead.
  client.recreate_collection(


True

In [3]:
prefix_path = "/home/inna/Рабочий стол/SneakerSearch/data/"

lamoda_data = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
lamoda_data

,brand,model,category,color,description,lamoda_photo,title_photo,path_to_lamoda_photo
0,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
1,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
2,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
3,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
4,Kappa,Kappa Кроссовки SIERRA MESH,Кроссовки,67545,NaN,https://a.lmcdn.ru/product/M/P/MP002XW12WY2_26...,lamoda_photos/Kappa_Kappa_Кроссовки_SIERRA_MES...,//a.lmcdn.ru/product/M/P/MP002XW12WY2_26197854...
...,...,...,...,...,...,...,...,...
2351,Baasploa,Baasploa Кроссовки,Низкие кроссовки,77514,Кроссовки выполнены из текстиля и подошва из E...,https://a.lmcdn.ru/product/M/P/MP002XW1E4IT_31...,lamoda_photos/Baasploa_Baasploa_Кроссовки__775...,//a.lmcdn.ru/product/M/P/MP002XW1E4IT_31820340...
2352,Baasploa,Baasploa Кроссовки,Низкие кроссовки,77514,Кроссовки выполнены из текстиля и подошва из E...,https://a.lmcdn.ru/product/M/P/MP002XW1E4IT_31...,lamoda_photos/Baasploa_Baasploa_Кроссовки__775...,//a.lmcdn.ru/product/M/P/MP002XW1E4IT_31820340...
2353,Baasploa,Baasploa Кроссовки,Низкие кроссовки,77514,Кроссовки выполнены из текстиля и подошва из E...,https://a.lmcdn.ru/product/M/P/MP002XW1E4IT_31...,lamoda_photos/Baasploa_Baasploa_Кроссовки__775...,//a.lmcdn.ru/product/M/P/MP002XW1E4IT_31820340...
2354,Baasploa,Baasploa Кроссовки,Низкие кроссовки,77514,Кроссовки выполнены из текстиля и подошва из E...,https://a.lmcdn.ru/product/M/P/MP002XW1E4IT_31...,lamoda_photos/Baasploa_Baasploa_Кроссовки__775...,//a.lmcdn.ru/product/M/P/MP002XW1E4IT_31820340...


In [4]:
# удаляем дубликаты
lamoda_data = lamoda_data.drop_duplicates(subset="title_photo")
print(len(lamoda_data))

2244


In [5]:
lamoda_data.iloc[3]["path_to_lamoda_photo"] == lamoda_data.iloc[6]["path_to_lamoda_photo"]

False

In [6]:
lamoda_data = lamoda_data.drop_duplicates(subset="path_to_lamoda_photo")
print(len(lamoda_data))

367


## CLIP emb

In [7]:
# инизацлизация модели для ембеддингов
model_name = "ViT-B/32" #338M params
model, preprocess = clip.load(model_name, device=device) 

In [8]:
img_path = prefix_path+lamoda_data.iloc[1]["title_photo"]
image = Image.open(img_path).convert("RGB")
preproc_image = preprocess(image).unsqueeze(0).to(device)
with torch.no_grad():
    image_emb = model.encode_image(preproc_image)
image_emb.shape

torch.Size([1, 512])

In [9]:
def get_image_embedding_clip(img_path):
    image = Image.open(img_path).convert('RGB')

    preproc_image = preprocess(image).unsqueeze(0).to(device)
    with torch.no_grad():
        image_emb = model.encode_image(preproc_image)
        
    # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
    image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [10]:
def create_db(
        client: QdrantClient,
        collection_name: str, 
        emb_dim: int,
        data: pd.DataFrame,
        get_image_embedding: callable):
    
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                 "photos": models.VectorParams(size=emb_dim, distance=models.Distance.COSINE)
            },
        )
        unprocessable = 0

        for idx, row in data.iterrows():
                img_path = prefix_path+lamoda_data.loc[idx]["title_photo"]
                try:
                    image_emb = get_image_embedding(img_path)

                    point = models.PointStruct(
                        id=idx, 
                        vector={
                             "photos" : image_emb.tolist()
                            }, 
                        payload={
                            "brand": row["brand"],
                            "model": row["model"],
                            "color": row["color"],
                            "path_to_photo": os.path.join(prefix_path, row["title_photo"]) # cохраняем путь для отображения!
                        }
                    )

                    client.upsert(
                        collection_name=collection_name, 
                        points=[point]
                        )
                except:
                    unprocessable+=1 
                    continue
        print("unprocessable: ", unprocessable)
    else:
        print("Коллекция уже существует")

In [12]:
emb_dim = 512
collection_name = "Sneakers_CLIP"

create_db(
        client,
        collection_name, 
        emb_dim,
        lamoda_data,
        get_image_embedding_clip)

unprocessable:  0


## Metrics CLIP

In [11]:
user_data = pd.read_csv(prefix_path+"user_data.csv", sep=";")
user_data = user_data.drop_duplicates()
user_data = user_data.drop_duplicates(subset="path_to_user_photo")
user_data = user_data.sample(frac=1, random_state=42).reset_index(drop=True) #shuffle data
print(len(user_data))

448


In [12]:
user_data

,brand,model,category,color,description,title_photo,path_to_title_photo,user_photo,path_to_user_photo
0,X-Plode,X-Plode Кроссовки,Низкие кроссовки,26114,NaN,//a.lmcdn.ru/product/M/P/MP002XW0VZPC_30825388...,lamoda_photos/X-Plode_X-Plode_Кроссовки__0.jpg,https://a.lmcdn.ru/photoreview/?key=a3f76460-f...,user_photos/X-Plode_X-Plode_Кроссовки__26114_1...
1,Founds,Founds Кроссовки,Низкие кроссовки,4993,NaN,//a.lmcdn.ru/product/R/T/RTLAER098401_32019644...,lamoda_photos/Founds_Founds_Кроссовки__0.jpg,https://a.lmcdn.ru/photoreview/?key=f4d48963-8...,user_photos/Founds_Founds_Кроссовки__4993_0.jpg
2,Lacoste,Lacoste Кеды BASESHOT PRO,Низкие кеды,33494,NaN,//a.lmcdn.ru/product/M/P/MP002XW1FS8B_26593567...,lamoda_photos/Lacoste_Lacoste_Кеды_BASESHOT_PR...,https://a.lmcdn.ru/photoreview/?key=f067d92c-f...,user_photos/Lacoste_Lacoste_Кеды_BASESHOT_PRO_...
3,X-Plode,X-Plode Кроссовки,Низкие кроссовки,78029,NaN,//a.lmcdn.ru/product/M/P/MP002XW0VZPD_30825747...,lamoda_photos/X-Plode_X-Plode_Кроссовки__0.jpg,https://a.lmcdn.ru/photoreview/?key=4195cf27-9...,user_photos/X-Plode_X-Plode_Кроссовки__78029_1...
4,Columbia,Columbia Кроссовки CRESTWOOD™,Низкие кроссовки,10099,Кроссовки с комбинированным верхом из натураль...,//a.lmcdn.ru/product/M/P/MP002XW02YFS_12636285...,lamoda_photos/Columbia_Columbia_Кроссовки_CRES...,https://a.lmcdn.ru/photoreview/?key=1b486e7b-f...,user_photos/Columbia_Columbia_Кроссовки_CRESTW...
...,...,...,...,...,...,...,...,...,...
443,Ecco,Ecco Кеды SOFT 7 W,Низкие кеды,547,NaN,//a.lmcdn.ru/product/M/P/MP002XW0S0Z2_11057679...,lamoda_photos/Ecco_Ecco_Кеды_SOFT_7_W_0.jpg,https://a.lmcdn.ru/photoreview/?key=decae862-b...,user_photos/Ecco_Ecco_Кеды_SOFT_7_W_0.jpg
444,Kappa,Kappa Кроссовки SELECTO MD,Кроссовки,93057,"Кроссовки выполнены из синтетической кожи, доп...",//a.lmcdn.ru/product/M/P/MP002XW0OTPY_22083371...,lamoda_photos/Kappa_Kappa_Кроссовки_SELECTO_MD...,https://a.lmcdn.ru/photoreview/?key=70c0ac26-b...,user_photos/Kappa_Kappa_Кроссовки_SELECTO_MD_2...
445,Karl Lagerfeld,Karl Lagerfeld Кеды,Низкие кеды,72503,NaN,//a.lmcdn.ru/product/M/P/MP002XW1DF5T_31481364...,lamoda_photos/Karl_Lagerfeld_Karl_Lagerfeld_Ке...,https://a.lmcdn.ru/photoreview/?key=757e82fc-0...,user_photos/Karl_Lagerfeld_Karl_Lagerfeld_Кеды...
446,Pierre Cardin,Pierre Cardin Кеды,Низкие кеды,53065,NaN,//a.lmcdn.ru/product/M/P/MP002XW1FBNU_26660772...,lamoda_photos/Pierre_Cardin_Pierre_Cardin_Кеды...,https://a.lmcdn.ru/photoreview/?key=a362e9b7-e...,user_photos/Pierre_Cardin_Pierre_Cardin_Кеды__...


In [13]:
test_data = user_data[-100:]
image_reference = dict(zip(test_data["path_to_user_photo"], test_data["model"]))

In [14]:
def count_metrics(
        collection_name: str,
        get_image_embedding: callable):

    recall_5 = 0
    recall_10 = 0
    accuracy = 0
    query_time = []
    N = 0
    unprocessable = []

    for image_path, reference_model in image_reference.items():
        try:
            start = time.perf_counter()
            query = get_image_embedding(prefix_path+image_path)
            candidates = client.query_points(
                                        collection_name=collection_name,
                                        query=query, 
                                        using="photos",
                                        limit=10,           
                                        with_payload=True   # Возвращаем бренд, модель и путь из CSV
                                    ).points
            end = time.perf_counter()
        
            results = [candidate.payload.get("model") for candidate in candidates]
            if reference_model in results:
                recall_10 += 1
            if reference_model in results[:5]:
                recall_5 += 1
            if reference_model == results[0]:
                accuracy += 1
            query_time.append(end-start)
            N +=1

        except:
            unprocessable.append(image_path)
            continue

    print("Unprocessable: ", len(unprocessable))

    return recall_10 *100 / N, recall_5 * 100 / N, accuracy * 100 / N, np.mean(query_time)

In [18]:
recall_10_clip, recall_5_clip, accuracy_clip, avg_query_time_clip = count_metrics(collection_name, get_image_embedding_clip)

Unprocessable:  0


In [19]:
df_data = {
    "Accuracy" : [accuracy_clip],
    "Recall@5": [recall_5_clip], 
    "Recall@10": [recall_10_clip],
    "Avg_query_time": [avg_query_time_clip]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,20.0,34.0,47.0,0.044016


## DINO emb

In [14]:
model_name = 'facebook/dinov2-large' #300M params


processor = AutoImageProcessor.from_pretrained(model_name)
model_dino = AutoModel.from_pretrained(model_name, device_map="auto")

Loading weights: 100%|██████████| 439/439 [00:00<00:00, 510.12it/s]


In [15]:
img_path = "/home/inna/Рабочий стол/SneakerSearch/scripts/lamoda_photos/Under_Armour_Under_Armour_Кроссовки_UA_W_Charged_P_0.jpg"
image = Image.open(img_path).convert("RGB")


inputs = processor(images=image, return_tensors="pt").to(device)
with torch.inference_mode():
    outputs = model_dino(**inputs)
    image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token

image_emb.shape

torch.Size([1, 1024])

In [16]:
def get_image_embedding_dino(img_path):
    image = Image.open(img_path).convert('RGB')
    
    preproc_image = processor(images=image, return_tensors="pt").to(device)
    with torch.inference_mode():
        outputs = model_dino(**preproc_image)
        image_emb = outputs.last_hidden_state[:, 0, :] # CLS-token
        # Нормализуем эмбеддинг (это важно для поиска в Qdrant)
        image_emb /= image_emb.norm(dim=-1, keepdim=True)
    
    return image_emb.cpu().numpy().flatten()

In [38]:
emb_dim = 1024
collection_name = "Sneakers_DINO"

create_db(
        client,
        collection_name, 
        emb_dim,
        lamoda_data,
        get_image_embedding_dino)

unprocessable:  0


In [39]:
recall_10_dino, recall_5_dino, accuracy_dino, avg_query_time_dino = count_metrics(collection_name, get_image_embedding_dino)

Unprocessable:  0


In [40]:
df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino],
    "Recall@5": [recall_5_clip, recall_5_dino], 
    "Recall@10": [recall_10_clip, recall_10_dino],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,20.0,34.0,47.0,0.044016
1,3.0,13.0,19.0,0.277981


## Mультивекторность Qdrant

**ЧТО:** 

В Qdrant реализованы два основных подхода к работе с несколькими векторами:
1. Именованные векторы (Named Vectors). Этот метод используется, когда один объект нужно описать разными характеристиками или модальностями. Если объединить эти эмбеддинги, получется много вариантов одного и того же объекта
2. Мультивекторные представления и Late Interaction. Более сложный механизм, предназначенный для моделей вроде ColBERT или ColPali, где один документ представляется не одним вектором, а набором векторов (например, по одному на каждый токен или фрагмент изображения). Если объединить эти эмбеддинги, получится представляемый ими объект.

MaxSim: Qdrant использует оператор MaxSim для вычисления сходства. Он находит наиболее похожий вектор в документе для каждого вектора в поисковом запросе и суммирует эти показатели

**ЗАЧЕМ:**

Можно же было бы создать для каждого ракурса свою точку в ВБД
1) Дедупликация в выдаче - не выдаст 3 одинаковые модели, потому что запрос похож на 3 ракурса одной модели. Всегда будут РАЗНЫЕ модели.
2) Управление данными (CRUD) - при необходимости обновить точку, надо будет обновить одну, а не искать все точки, принадлежащие этой модели
3) Производительность и индексы - используется специальный тип сжатия и индексации. В случае с max_sim Qdrant оптимизирует поиск так, чтобы не сравнивать запрос с каждым вектором в лоб, а использовать аппроксимацию.


In [15]:
lamoda_data_racurses = pd.read_csv(prefix_path+"lamoda_data.csv", sep=";")
lamoda_data_racurses = lamoda_data_racurses.drop_duplicates(subset="title_photo")
print(len(lamoda_data_racurses))

2244


In [16]:
grouped = lamoda_data_racurses.groupby(['brand', 'model', 'color'])['title_photo'].apply(list).reset_index()
print(grouped)

            brand                                       model  color  \
0             361                    361 Кроссовки CENTAURI 2  17296   
1          4forms                                4forms Кеды    3588   
2            ACBC  ACBC Кроссовки LOW TOP WOMAN ECO MATERIALS  40124   
3           ASICS                   ASICS Кроссовки FUJISPEED  42637   
4           ASICS               ASICS Кроссовки GEL-KAYANO 32  46422   
..            ...                                         ...    ...   
374  adidas YEEZY   adidas YEEZY Кроссовки YEEZY BOOST 350 V2  21383   
375      s.Oliver                         s.Oliver Кроссовки   54994   
376          Араз                             Араз Кроссовки   10798   
377      Два Мяча                              Два Мяча Кеды   29971   
378      Два Мяча                              Два Мяча Кеды   67693   

                                           title_photo  
0    [lamoda_photos/361_361_Кроссовки_CENTAURI_2_17...  
1    [lamoda_photos/4

In [31]:
def create_multivector_db(
        client: QdrantClient,
        collection_name: str, 
        emb_dim: int,
        data: pd.DataFrame,
        get_image_embedding: callable):
    
    unprocessable = 0
    
    if not client.collection_exists(collection_name):
        client.create_collection(
            collection_name=collection_name,
            vectors_config={
                "photos": models.VectorParams(
                    size=emb_dim, 
                    distance=models.Distance.COSINE,
                    multivector_config=models.MultiVectorConfig(
                        comparator=models.MultiVectorComparator.MAX_SIM #позволяет хранить список векторов под одним именем
                    ) 
                )
            }
        )

        for idx, row in data.iterrows():
            try:
                multivector = []
                for racurs in row["title_photo"]:
                    try:
                        img_path = prefix_path + racurs
                        img_emb = get_image_embedding(img_path)
                        multivector.append(img_emb.tolist())
                    except:
                        continue

                point = models.PointStruct(
                    id=idx, 
                    vector={
                        "photos": multivector
                        }, 
                    payload={
                        "brand": row["brand"],
                        "model": row["model"],
                        "color": row["color"],
                        "path_to_photo": os.path.join(prefix_path, row["title_photo"][0]) # cохраняем путь для отображения!
                    }
                )

                client.upsert(
                    collection_name=collection_name, 
                    points=[point]
                    )
                    
            except: 
                unprocessable+=1
                continue
        print("unprocessable: ", unprocessable)
    else:
        print("Коллекция уже существует")

In [34]:
collection_name = "Sneakers_CLIP_multivector"

create_multivector_db(
    client,
    collection_name, 
    emb_dim = 512,
    data = grouped,
    get_image_embedding=get_image_embedding_clip)

unprocessable:  0


In [ ]:
recall_10_clip_multivectors, recall_5_clip_multivectors, accuracy_clip_multivectors, avg_query_time_clip_multivectors = count_metrics(collection_name, get_image_embedding_clip)

df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino, accuracy_clip_multivectors],
    "Recall@5": [recall_5_clip, recall_5_dino, recall_5_clip_multivectors], 
    "Recall@10": [recall_10_clip, recall_10_dino, recall_10_clip_multivectors],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino, avg_query_time_clip_multivectors]
}
  
table = pd.DataFrame(data=df_data)
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
0,20.0,34.0,47.0,0.044016
1,3.0,13.0,19.0,0.277981
2,23.0,39.0,45.0,0.034411


In [36]:
collection_name = "Sneakers_DINO_multivector"

create_multivector_db(
    client,
    collection_name, 
    emb_dim = 1024,
    data = grouped,
    get_image_embedding=get_image_embedding_dino)

unprocessable:  0


In [ ]:
recall_10_dino_multivectors, recall_5_dino_multivectors, accuracy_dino_multivectors, avg_query_time_dino_multivectors = count_metrics(collection_name, get_image_embedding_dino)

df_data = {
    "Accuracy" : [accuracy_clip, accuracy_dino, accuracy_clip_multivectors, accuracy_dino_multivectors],
    "Recall@5": [recall_5_clip, recall_5_dino, recall_5_clip_multivectors, recall_5_dino_multivectors], 
    "Recall@10": [recall_10_clip, recall_10_dino, recall_10_clip_multivectors, recall_10_dino_multivectors],
    "Avg_query_time": [avg_query_time_clip, avg_query_time_dino, avg_query_time_clip_multivectors, avg_query_time_dino_multivectors]
}
  
table = pd.DataFrame(data=df_data, index=["CLIP-ViT", "DINOv2", "CLIP_multivector", "DINO_multivector"])
table

,Accuracy,Recall@5,Recall@10,Avg_query_time
CLIP-ViT,20.0,34.0,47.0,0.044016
DINOv2,3.0,13.0,19.0,0.277981
CLIP_multivector,23.0,39.0,45.0,0.034411
DINO_multivector,6.0,13.0,18.0,0.274234


In [43]:
pd.options.display.float_format = '{:.2f}'.format
pd.DataFrame(
    data={
        "CLIP-ViT" : [accuracy_clip, recall_5_clip, recall_10_clip, avg_query_time_clip, 15.4],
        "DINOv2": [accuracy_dino, recall_5_dino, recall_10_dino, avg_query_time_dino, 76.5],
        "CLIP_multivector" : [accuracy_clip_multivectors, recall_5_clip_multivectors, recall_10_clip_multivectors,avg_query_time_clip_multivectors, 100.1],
        "DINO_multivector": [accuracy_dino_multivectors, recall_5_dino_multivectors, recall_10_dino_multivectors, avg_query_time_dino_multivectors, 279.3]
    },
    index = ["Accuracy", "Recall@5", "Recall@10", "Avg_query_time, s", "Create_DB_time, s"]
)


,CLIP-ViT,DINOv2,CLIP_multivector,DINO_multivector
Accuracy,20.00,3.00,23.00,6.00
Recall@5,34.00,13.00,39.00,13.00
Recall@10,47.00,19.00,45.00,18.00
"Avg_query_time, s",0.04,0.28,0.03,0.27
"Create_DB_time, s",15.40,76.50,100.10,279.30


## Mulvis (IVF index)

In [44]:
milvus_client = MilvusClient("./milvus_local.db")

In [45]:
def create_milvus_bd(
        collection_name: str, 
        emb_dim: int, 
        data: pd.DataFrame,
        get_image_embedding: callable):
    # Создаем схему и коллекцию для CLIP
    schema = milvus_client.create_schema(auto_id=False)
    schema.add_field(field_name="id", datatype=DataType.INT64, is_primary=True)
    schema.add_field(field_name="vector", datatype=DataType.FLOAT_VECTOR, dim=emb_dim)
    schema.add_field(field_name="brand", datatype=DataType.VARCHAR, max_length=500)
    schema.add_field(field_name="model", datatype=DataType.VARCHAR, max_length=500)
    schema.add_field(field_name="path_to_photo", datatype=DataType.VARCHAR, max_length=500)

    milvus_client.create_collection(collection_name=collection_name, schema=schema)

    # Создаем IVF индекс
    index_params = milvus_client.prepare_index_params()
    index_params.add_index(
        field_name="vector",
        index_type="IVF_FLAT",
        metric_type="COSINE",
        params={"nlist": 1024}
    )
    milvus_client.create_index(collection_name=collection_name, index_params=index_params)

    #Добавление данных
    unprocessable = 0
    for idx, row in data.iterrows():
        img_path = prefix_path+row["title_photo"]
        try:
            image_emb = get_image_embedding(img_path)
            image_emb = image_emb.astype('float32')

            res = milvus_client.insert(
                collection_name=collection_name,
                data=[
                    {
                        "id": idx, 
                        "vector": image_emb,
                        "brand": row["brand"],
                        "model": row["model"],
                        "path_to_photo": os.path.join(prefix_path, row["title_photo"][0]) 
                        }
                        ]
            )
        except:
            unprocessable+=1
            continue
    print("unprocessable: ", unprocessable)

In [50]:
collection_name = "CLIP_IVF"
emb_dim = 512

create_milvus_bd(collection_name, emb_dim, lamoda_data, get_image_embedding_clip)

unprocessable:  0


In [51]:
#milvus_client.describe_collection(collection_name)
#milvus_client.drop_collection(collection_name=collection_name)

stats = milvus_client.get_collection_stats(collection_name=collection_name)
print(f"Количество точек: {stats['row_count']}")

Количество точек: 367


In [54]:
def count_metrics_milvus(collection_name, get_image_embedding):

    recall_5 = 0
    recall_10 = 0
    accuracy = 0
    query_time = []
    N = 0
    unprocessable = []

    for image_path, reference_model in image_reference.items():
        try:
            start = time.perf_counter()
            query = get_image_embedding(prefix_path+image_path)
            query = query.astype('float32')
            candidates = milvus_client.search(
                                        collection_name=collection_name,
                                        data=[query], 
                                        limit=10,
                                        search_params={"params": {"nprobe": 10}},
                                         output_fields=["brand", "model"]
                                    )
            end = time.perf_counter()
        
            results = [candidate["model"] for candidate in candidates[0]]
            if reference_model in results:
                recall_10 += 1
            if reference_model in results[:5]:
                recall_5 += 1
            if reference_model == results[0]:
                accuracy += 1
            query_time.append(end-start)
            N +=1

        except:
            unprocessable.append(image_path)
            continue

    print("Unprocessable: ", len(unprocessable))

    return recall_10 *100 / N, recall_5 * 100 / N, accuracy * 100 / N, np.mean(query_time)

In [55]:
recall_10_clip_ivf, recall_5_clip_ivf, accuracy_clip_ivf, avg_query_time_clip_ivf = count_metrics_milvus(collection_name, get_image_embedding_clip)

I0511 14:44:16.265460  732400 chttp2_transport.cc:1369] unix:/tmp/tmptmi3wnmz_milvus_local.db.sock: Got goaway [11] err=UNAVAILABLE:GOAWAY received; Error code: 11; Debug Text: too_many_pings {grpc_status:14, http2_error:11}
E0511 14:44:16.265534  732400 chttp2_transport.cc:1401] unix:/tmp/tmptmi3wnmz_milvus_local.db.sock: Received a GOAWAY with error code ENHANCE_YOUR_CALM and debug data equal to "too_many_pings". Current keepalive time (before throttling): 10000ms


Unprocessable:  0


In [56]:
pd.options.display.float_format = '{:.2f}'.format
pd.DataFrame(
    data={
        "CLIP-ViT" : [accuracy_clip, recall_5_clip, recall_10_clip, avg_query_time_clip, 15.4],
        "DINOv2": [accuracy_dino, recall_5_dino, recall_10_dino, avg_query_time_dino, 76.5],
        "CLIP_multivector" : [accuracy_clip_multivectors, recall_5_clip_multivectors, recall_10_clip_multivectors,avg_query_time_clip_multivectors, 100.1],
        "DINO_multivector": [accuracy_dino_multivectors, recall_5_dino_multivectors, recall_10_dino_multivectors, avg_query_time_dino_multivectors, 279.3],
        "CLIP_Milvus_IVF" : [accuracy_clip_ivf, recall_5_clip_ivf, recall_10_clip_ivf, avg_query_time_clip_ivf, 15.0]
    },
    index = ["Accuracy", "Recall@5", "Recall@10", "Avg_query_time, s", "Create_DB_time, s"]
)

,CLIP-ViT,DINOv2,CLIP_multivector,DINO_multivector,CLIP_Milvus_IVF
Accuracy,20.00,3.00,23.00,6.00,20.00
Recall@5,34.00,13.00,39.00,13.00,34.00
Recall@10,47.00,19.00,45.00,18.00,47.00
"Avg_query_time, s",0.04,0.28,0.03,0.27,0.03
"Create_DB_time, s",15.40,76.50,100.10,279.30,15.00


In [58]:
# DINOv2 + Milvus_IVF

collection_name = "DINO_IVF"
emb_dim = 1024

create_milvus_bd(collection_name, emb_dim, lamoda_data, get_image_embedding_dino)

unprocessable:  0


In [59]:
stats = milvus_client.get_collection_stats(collection_name=collection_name)
print(f"Количество точек: {stats['row_count']}")

Количество точек: 367


In [61]:
recall_10_dino_ivf, recall_5_dino_ivf, accuracy_dino_ivf, avg_query_time_dino_ivf = count_metrics_milvus(collection_name, get_image_embedding_dino)

Unprocessable:  0


In [62]:
pd.DataFrame(
    data={
        "CLIP-ViT" : [accuracy_clip, recall_5_clip, recall_10_clip, avg_query_time_clip, 15.4],
        "DINOv2": [accuracy_dino, recall_5_dino, recall_10_dino, avg_query_time_dino, 76.5],
        "CLIP_multivector" : [accuracy_clip_multivectors, recall_5_clip_multivectors, recall_10_clip_multivectors,avg_query_time_clip_multivectors, 100.1],
        "DINO_multivector": [accuracy_dino_multivectors, recall_5_dino_multivectors, recall_10_dino_multivectors, avg_query_time_dino_multivectors, 279.3],
        "CLIP_Milvus_IVF" : [accuracy_clip_ivf, recall_5_clip_ivf, recall_10_clip_ivf, avg_query_time_clip_ivf, 15.0],
        "DINO_Milvus_IVF" : [accuracy_dino_ivf, recall_5_dino_ivf, recall_10_dino_ivf, avg_query_time_dino_ivf, 22.6]
    },
    index = ["Accuracy", "Recall@5", "Recall@10", "Avg_query_time, s", "Create_DB_time, s"]
)

,CLIP-ViT,DINOv2,CLIP_multivector,DINO_multivector,CLIP_Milvus_IVF,DINO_Milvus_IVF
Accuracy,20.00,3.00,23.00,6.00,20.00,3.00
Recall@5,34.00,13.00,39.00,13.00,34.00,13.00
Recall@10,47.00,19.00,45.00,18.00,47.00,19.00
"Avg_query_time, s",0.04,0.28,0.03,0.27,0.03,0.08
"Create_DB_time, s",15.40,76.50,100.10,279.30,15.00,22.60


## Preprocessing

Для фильтрации фотографий, не содержащих кроссовки проверяется 2 варианта:

1) Использование модели для детекции объектов YOLOS, предобученной на датасете fashionpedia (46781 images, 342182 bounding-boxes, num_classes=46, names=46) с элементами одежды (в том числе обувь, class=5 "shoe"). Мотивация применения подобных моделей обучсловена тем, что на выходе получаем не только ответ на вопрос "присутствуют ли кроссовки на фотографии?", но и bounding-box, позволяющий обрезать лишние части картинки и далее работать непосредственно с изображением кроссовка.

Без дообучения, данная модель не распознала ни одного кроссовка как на бытовых пользовательских фотографиях, так и на студийных, несмотря на снижение порога детекции до 0.1. Вследствие чего такой подход может быть применен только после дообучения на размеченных данных. 

2) Использование легкой сверточной модели Mobilenet, с заменой выходного слоя на классификацию на 2 класса "на фото есть кроссовки" и "на фото нет кроссовок". Эмпирически определен уровень вероятности 0.4 при котором модель в 100% случаев определяет на студийных фото наличие кроссовок, и в 95% случаев на бытовых пользовательских фотографиях. В 5% отсеяных фото попали фотографии подошвы, макрофото части носка, либо фото, на которых кроссовки занимают очень маленький процент изображения.  

In [ ]:
# YOLOS_FT_FASHIONPEDIA

from transformers import  AutoModelForObjectDetection

object_detection_model_name = "valentinafevu/yolos-fashionpedia"
image_processor = AutoImageProcessor.from_pretrained(object_detection_model_name)
model_yolo = AutoModelForObjectDetection.from_pretrained(object_detection_model_name)

Loading weights: 100%|██████████| 212/212 [00:00<00:00, 6825.61it/s]


In [113]:
model_yolo.to(device)

YolosForObjectDetection(
  (vit): YolosModel(
    (embeddings): YolosEmbeddings(
      (patch_embeddings): YolosPatchEmbeddings(
        (projection): Conv2d(3, 384, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
      (interpolation): InterpolateInitialPositionEmbeddings()
    )
    (encoder): YolosEncoder(
      (layer): ModuleList(
        (0-11): 12 x YolosLayer(
          (attention): YolosAttention(
            (attention): YolosSelfAttention(
              (query): Linear(in_features=384, out_features=384, bias=True)
              (key): Linear(in_features=384, out_features=384, bias=True)
              (value): Linear(in_features=384, out_features=384, bias=True)
            )
            (output): YolosSelfOutput(
              (dense): Linear(in_features=384, out_features=384, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): YolosIntermediate(
            (dense

In [ ]:
is_sneakers = 0
no_sneakers = 0
for image_path, reference_model in image_reference.items():
    img_path = prefix_path+image_path
    image = Image.open(img_path).convert('RGB')

    with torch.no_grad():
        inputs = image_processor(images=[image], return_tensors="pt")
        outputs = model_yolo(**inputs.to(device))
        target_sizes = torch.tensor([[image.size[1], image.size[0]]])
        results = image_processor.post_process_object_detection(outputs, threshold=0.1, target_sizes=target_sizes)[0]
    if 5 in results["labels"]:
        is_sneakers += 1
    else: 
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  0  no_sneakers:  100


In [143]:
is_sneakers = 0
no_sneakers = 0
for idx, row in lamoda_data[:100].iterrows():
    img_path = prefix_path+row["title_photo"]
    image = Image.open(img_path).convert('RGB')
   
    with torch.no_grad():
        inputs = image_processor(images=[image], return_tensors="pt")
        outputs = model_yolo(**inputs.to(device))
        target_sizes = torch.tensor([[image.size[1], image.size[0]]])
        results = image_processor.post_process_object_detection(outputs, threshold=0.1, target_sizes=target_sizes)[0]
    if 5 in results["labels"]:
        is_sneakers += 1
    else: 
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  0  no_sneakers:  100


In [67]:
# MOBILENET_V3_LARGE

import torchvision.models as models

mobilenet = models.mobilenet_v3_large(weights="IMAGENET1K_V2")

In [74]:
mobilenet.to(device)

MobileNetV3(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
      (2): Hardswish()
    )
    (1): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=16, bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
          (2): ReLU(inplace=True)
        )
        (1): Conv2dNormActivation(
          (0): Conv2d(16, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(16, eps=0.001, momentum=0.01, affine=True, track_running_stats=True)
        )
      )
    )
    (2): InvertedResidual(
      (block): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 64, kernel_size=(1, 1), stride=(1, 1), bi

In [69]:
for param in mobilenet.parameters():
    param.requires_grad = False

# В MobileNetV3 классификатор находится в атрибуте .classifier
# Последний слой там — линейный под индексом [1]
num_features = mobilenet.classifier[3].in_features
mobilenet.classifier[3] = torch.nn.Linear(num_features, 2) # 2 класса: "есть кроссовки" и "нет"

print(mobilenet.classifier)

Sequential(
  (0): Linear(in_features=960, out_features=1280, bias=True)
  (1): Hardswish()
  (2): Dropout(p=0.2, inplace=True)
  (3): Linear(in_features=1280, out_features=2, bias=True)
)


In [75]:
from torchvision import transforms

#трансформации для mobilenet
mobilenet_preprocess = transforms.Compose([
    transforms.Resize(256),                  # изменяем размер меньшей стороны до 256
    transforms.CenterCrop(224),              # обрезаем центр до 224x224
    transforms.ToTensor(),                   # переводим в тензор и нормализуем в [0, 1]
    transforms.Normalize(                    # стандартная нормализация для ImageNet
        mean=[0.485, 0.456, 0.406], 
        std=[0.229, 0.224, 0.225]
    ),
])


def sneakers_classifier(img_path):

    image = Image.open(img_path).convert("RGB") 
    prerpoc_image = mobilenet_preprocess(image)                     
    input = torch.unsqueeze(prerpoc_image, 0) 
    input = input.to(device)
         
    with torch.no_grad():
        output = mobilenet(input)

    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    #print(f"Вероятность кроссовок: {probabilities[0].item():.2%}")
    return probabilities[0].item()

In [76]:
# юзерские фото с кроссовками

mobilenet.eval() 

NO_SNEAKERS=[]
is_sneakers = 0
no_sneakers = 0
for image_path, reference_model in image_reference.items():
    img_path = prefix_path+image_path
    sneakers_proba = sneakers_classifier(img_path)
    if sneakers_proba >= 0.4:
        is_sneakers += 1
    else:
        NO_SNEAKERS.append(image_path)
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  95  no_sneakers:  5


In [77]:
NO_SNEAKERS

['user_photos/adidas_Originals_adidas_Originals_Кеды_GAZELLE_LO__2.jpg',
 'user_photos/adidas_adidas_Кроссовки_RUN_60s_4.0_0.jpg',
 'user_photos/adidas_adidas_Кроссовки_ADIZERO_EVO_SL_EXO_W_0.jpg',
 'user_photos/adidas_adidas_Кроссовки_Barricade_14_W_0.jpg',
 'user_photos/Salamander_Salamander_Кеды_Lamoda_Online_Exclusive_0.jpg']

In [78]:
# студийные фото кроссовок

NO_SNEAKERS = []

is_sneakers = 0
no_sneakers = 0
for idx, row in lamoda_data[:100].iterrows():
    img_path = prefix_path+row["title_photo"]
    sneakers_proba = sneakers_classifier(img_path)
    if sneakers_proba >= 0.4:
        is_sneakers += 1
    else:
        NO_SNEAKERS.append(img_path)
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  100  no_sneakers:  0


In [81]:
# все не кроссовки

IS_SNEAKERS=[]
is_sneakers = 0
no_sneakers = 0
for file in os.listdir("/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers"):
    img_path = prefix_path+"No_sneakers/"+file
    sneakers_proba = sneakers_classifier(img_path)
    if sneakers_proba >= 0.4:
        is_sneakers += 1
        IS_SNEAKERS.append(img_path)
    else:
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  185  no_sneakers:  14


In [82]:
IS_SNEAKERS

['/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/cjrzju9pwjlb1.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/sport_17935.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/kot_Novyj_god_sharik_5529.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/zhivotnye_kot_nogi_11902.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/1646557638_28-bigfoto-name-p-komnati-v-kvartire-realnie-prostie-deshevi-62.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/nogi_obuv_1386.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/nogi_9764.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/kniga_nogi_8055.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/muzhchina_sport_serfing_1334.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/7957394408.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/92045f734fa8ec5e3d076ba6cd6054a0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/d

## FT

In [17]:
other_data = ["No_sneakers/" + filename for filename in os.listdir("/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers")]
len(other_data)

199

In [18]:
import random
indices = list(range(len(other_data)))
random.seed(42)
random.shuffle(indices)

train_data_other = [other_data[i] for i in indices[:-100]]
test_data_other = [other_data[i] for i in indices[-100:]]

train_data_sneakers = user_data[:-100].path_to_user_photo.to_list()
test_data = user_data[-100:].path_to_user_photo.to_list()

In [85]:
train_transforms = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

test_transforms = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

In [86]:
from torch.utils.data import Dataset, DataLoader

class SneakersDataset(Dataset):
    def __init__(self, paths_sneakers, paths_other, transform=None):
        # кроссовки - 0, не кроссовки - 1 
        self.image_paths = paths_sneakers + paths_other
        self.labels = [0] * len(paths_sneakers) + [1] * len(paths_other)
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        label = self.labels[idx]
        
        # Загружаем картинку и переводим в RGB
        image = Image.open(prefix_path + img_path).convert("RGB")
        
        if self.transform:
            image = self.transform(image)
            
        return image, label

In [87]:
train_dataset = SneakersDataset(train_data_sneakers, train_data_other, transform=train_transforms)
test_dataset = SneakersDataset(test_data, test_data_other, transform=train_transforms)

# 4. Создаем DataLoader (теперь их можно подавать в цикл обучения)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [88]:
criterion = torch.nn.CrossEntropyLoss()
# оптимизируем только параметры классификатора
optimizer = torch.optim.Adam(mobilenet.classifier[3].parameters(), lr=0.001)

mobilenet.to(device)

best_accuracy = 0
ft_accuracy = []

for epoch in range(20):

    mobilenet.train()

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = mobilenet(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    mobilenet.eval()

    correct = 0
    total = 0
    
    with torch.no_grad():
        for inputs, labels in test_loader: 
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = mobilenet(inputs)
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_acc = 100 * correct / total
    print(f'Точность на валидации, {epoch} эпоха: {val_acc}%')

    ft_accuracy.append(val_acc)
    if val_acc>best_accuracy:
        torch.save(mobilenet.state_dict(), '/home/inna/Рабочий стол/SneakerSearch/models/best_mobilenet_sneakers.pth')

Точность на валидации, 0 эпоха: 53.5%
Точность на валидации, 1 эпоха: 56.5%
Точность на валидации, 2 эпоха: 59.0%
Точность на валидации, 3 эпоха: 66.5%
Точность на валидации, 4 эпоха: 69.5%
Точность на валидации, 5 эпоха: 76.5%
Точность на валидации, 6 эпоха: 78.0%
Точность на валидации, 7 эпоха: 82.5%
Точность на валидации, 8 эпоха: 84.5%
Точность на валидации, 9 эпоха: 83.5%
Точность на валидации, 10 эпоха: 83.0%
Точность на валидации, 11 эпоха: 84.5%
Точность на валидации, 12 эпоха: 86.0%
Точность на валидации, 13 эпоха: 88.0%
Точность на валидации, 14 эпоха: 87.0%
Точность на валидации, 15 эпоха: 89.5%
Точность на валидации, 16 эпоха: 84.0%
Точность на валидации, 17 эпоха: 83.5%
Точность на валидации, 18 эпоха: 87.5%
Точность на валидации, 19 эпоха: 88.0%


In [ ]:
state_dict = torch.load('/home/inna/Рабочий стол/SneakerSearch/models/best_mobilenet_sneakers.pth')

mobilenet.load_state_dict(state_dict)
mobilenet.eval()

<All keys matched successfully>

In [104]:
# студийные кроссовки
NO_SNEAKERS=[]
is_sneakers = 0
no_sneakers = 0
for idx, row in lamoda_data[:100].iterrows():
    img_path = prefix_path+row["title_photo"]
    sneakers_proba = sneakers_classifier(img_path)
    if sneakers_proba >= 0.4:
        is_sneakers += 1
    else:
        NO_SNEAKERS.append(img_path)
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  95  no_sneakers:  5


In [105]:
NO_SNEAKERS

['/home/inna/Рабочий стол/SneakerSearch/data/lamoda_photos/Matrix_Sport_Matrix_Sport_Кеды_Iconic_44186_0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/lamoda_photos/Covani_Covani_Кеды__1139_0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/lamoda_photos/Reversal_Reversal_Кроссовки__16465_0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/lamoda_photos/Lacoste_Lacoste_Кроссовки_ELITE_ACTIVE_52450_0.jpg',
 "/home/inna/Рабочий стол/SneakerSearch/data/lamoda_photos/O'STIN_O'STIN_Кроссовки__99508_0.jpg"]

In [106]:
# кроссовки
NO_SNEAKERS=[]
is_sneakers = 0
no_sneakers = 0
for image_path, reference_model in image_reference.items():
    img_path = prefix_path+image_path
    sneakers_proba = sneakers_classifier(img_path)
    if sneakers_proba >= 0.4:
        is_sneakers += 1
    else:
        NO_SNEAKERS.append(img_path)
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  96  no_sneakers:  4


In [107]:
NO_SNEAKERS

['/home/inna/Рабочий стол/SneakerSearch/data/user_photos/Cotton_Belt_Cotton_Belt_Кеды_Nc_0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/user_photos/Reebok_Reebok_Кеды_CLUB_C_85_VINTAGE_21128_0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/user_photos/Flower_Mountain_Flower_Mountain_Кроссовки_WAVE_2.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/user_photos/Nike_Nike_Кроссовки_W_ZOOM_FLY_6_0.jpg']

In [103]:
# все не кроссовки

IS_SNEAKERS=[]
is_sneakers = 0
no_sneakers = 0
for file in os.listdir("/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers"):
    img_path = prefix_path+"No_sneakers/"+file
    sneakers_proba = sneakers_classifier(img_path)
    if sneakers_proba >= 0.4:
        is_sneakers += 1
        IS_SNEAKERS.append(img_path)
    else:
        no_sneakers += 1
print("is_sneakers: ", is_sneakers, " no_sneakers: ", no_sneakers)

is_sneakers:  29  no_sneakers:  170


In [263]:
IS_SNEAKERS

['/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/nogi_9764.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/7957394408.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/92045f734fa8ec5e3d076ba6cd6054a0.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/Ue929f670f945415490bdc1f65c8c3c15D.jpg_960x960.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/photo-2023-09-14-19-57-46.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/opening-box-new-leather-boots-person-cardboard-shoe-inside-concept-shopping-footwear-purchase-retail-packaging-444795069.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/9eb967890cd2e56df6dfa33429418f0d.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/8702180870.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/11.jpg',
 '/home/inna/Рабочий стол/SneakerSearch/data/No_sneakers/ab09d6880e330d8291d7f149e29b6484e342807e_1024_1024.jpeg',
 '/home/inna/Рабочий ст

## FT CLIP

In [17]:
from collections import defaultdict

lamoda = defaultdict(list)
for idx, row in lamoda_data_racurses.iterrows():
    lamoda[f'{row["model"]}_{row["color"]}'].append(row["title_photo"])
print("Студийных фото ", len(lamoda))

user = defaultdict(list)
for idx, row in user_data.iterrows():
    user[f'{row["model"]}_{row["color"]}'].append(row["path_to_user_photo"])
print("Пользовательских фото ", len(user))

Студийных фото  379
Пользовательских фото  185


In [ ]:
from datasets import Dataset
from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments


common_models = set(lamoda.keys()).intersection(set(user.keys()))

pairs = []
for model_id in common_models:
    for user_photo in user[f"{model_id}"]:
        lamoda_photo = random.choice(lamoda[f"{model_id}"])
        pairs.append((user_photo, lamoda_photo))

# 1. Подготовка данных в формате словаря (только пути!)
train_data_dict = {
    "image_1": [prefix_path + p[0] for p in pairs],
    "image_2": [prefix_path + p[1] for p in pairs]
}
hf_dataset = Dataset.from_dict(train_data_dict)

model = SentenceTransformer('clip-ViT-B-32', device=device)
train_loss = losses.MultipleNegativesRankingLoss(model)

# 2. Настройка аргументов обучения
args = SentenceTransformerTrainingArguments(
 output_dir='/home/inna/Рабочий стол/SneakerSearch/models/ft_clip_sneakers',
    num_train_epochs=5,          
    per_device_train_batch_size=16, # Уменьшим батч для стабильности
    warmup_steps=20,
    fp16=True,
    save_steps=100,
    logging_steps=10,
    dataloader_num_workers=0,      
    group_by_length=False,        
    report_to="none"  
)

# 3. Создание тренера
trainer = SentenceTransformerTrainer(
    model=model,
    args=args,
    train_dataset=hf_dataset,
    loss=train_loss,
)

# 4. Запуск (теперь система не должна виснуть)
trainer.train()


/tmp/ipykernel_763905/87939775.py:2: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, losses, SentenceTransformerTrainer, SentenceTransformerTrainingArguments
Loading weights: 100%|██████████| 398/398 [00:00<00:00, 4643.29it/s]
CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:




# # class ImagePairDataset(Dataset):
# #     def __init__(self, pairs, prefix_path):
# #         self.pairs = pairs
# #         self.prefix_path = prefix_path

# #     def __len__(self):
# #         return len(self.pairs)

# #     def __getitem__(self, idx):
# #         user_path, studio_path = self.pairs[idx]
# #         # Открываем картинки только СЕЙЧАС (on-demand)
# #         img1 = Image.open(self.prefix_path + user_path).convert('RGB')
# #         img2 = Image.open(self.prefix_path + studio_path).convert('RGB')
# #         return InputExample(texts=[img1, img2])

# common_models = set(lamoda.keys()).intersection(set(user.keys()))

# # pairs = []
# # for model_id in common_models:
# #     for user_photo in user[f"{model_id}"]:
# #         lamoda_photo = random.choice(lamoda[f"{model_id}"])
# #         pairs.append((user_photo, lamoda_photo))

# # train_dataset = ImagePairDataset(pairs, prefix_path)
# # train_dataloader = DataLoader(train_dataset, shuffle=True, batch_size=32, num_workers=4)

# train_examples = []

# # Сопоставляем каждое пользовательское фото со студийным фото этой же модели
# for model_id in common_models:
#     for user_photo in user[f"{model_id}"]:
#         random_idx = random.randint(0, len(lamoda[f"{model_id}"])-1)
#         lamoda_photo = lamoda[f"{model_id}"][random_idx]
#         # Создаем пару положительных примеров
#         train_examples.append(InputExample(texts=[
#             Image.open(prefix_path + user_photo), 
#             Image.open(prefix_path + lamoda_photo)
#         ]))

# train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [22]:
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import Dataset, DataLoader

common_models = set(lamoda.keys()).intersection(set(user.keys()))

train_examples = []
for model_id in common_models:
    for user_photo in user[model_id]:
        studio_photo = random.choice(lamoda[model_id])
      
        train_examples.append(InputExample(texts=[
            prefix_path + user_photo, 
            prefix_path + studio_photo
        ]))

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

/tmp/ipykernel_747389/3660221959.py:1: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import SentenceTransformer, InputExample, losses


In [ ]:
train_loss = losses.MultipleNegativesRankingLoss(model)

model = SentenceTransformer('clip-ViT-B-32', device=device)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=10, 
    warmup_steps=20,
    output_path='/home/inna/Рабочий стол/SneakerSearch/models/ft_clip_sneakers'
)

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 4187.37it/s]
CLIPModel LOAD REPORT from: sentence-transformers/clip-ViT-B-32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_image_embedding(img_path):

    image = Image.open(img_path).convert('RGB')

    with torch.no_grad():
        image_emb = model.encode(
            image,
            convert_to_tensor=True,
            normalize_embeddings=True
            )
    
    return image_emb.cpu().numpy().flatten()

In [ ]:
collection_name = "Sneakers_CLIP_FT"
emb_dim = 512

create_db(
    client,
    collection_name,
    emb_dim,
    lamoda_data
)

In [ ]:
recall_10_clip_ft, recall_5_clip_ft, accuracy_clip_ft, avg_query_time_clip_ft = count_metrics(collection_name)

pd.DataFrame(
    data={
        "CLIP-ViT" : [accuracy_clip, recall_5_clip, recall_10_clip, avg_query_time_clip, 11.3],
        "DINOv2": [accuracy_dino, recall_5_dino, recall_10_dino, avg_query_time_dino, 19.3],
        "CLIP_multivector" : [accuracy_clip_multivectors, recall_5_clip_multivectors, recall_10_clip_multivectors,avg_query_time_clip_multivectors, 264.3],
        "DINO_multivector": [accuracy_dino_multivectors, recall_5_dino_multivectors, recall_10_dino_multivectors, avg_query_time_dino_multivectors, 468.6],
        "CLIP_Milvus_IVF" : [accuracy_clip_ivf, recall_5_clip_ivf, recall_10_clip_ivf, avg_query_time_clip_ivf, 11.6],
        "DINO_Milvus_IVF" : [accuracy_dino_ivf, recall_5_dino_ivf, recall_10_dino_ivf, avg_query_time_dino_ivf, 53],
        "CLIP_FT" : [accuracy_clip_ft, recall_5_clip_ft, recall_10_clip_ft, avg_query_time_clip_ft, ]
    },
    index = ["Accuracy", "Recall@5", "Recall@10", "Avg_query_time, s", "Create_DB_time, s"]
)